In [ ]:
!pip install wandb

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import wandb
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghosalsohom2003 (ghosalsohom2003-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Model

In [ ]:
x = torch.linspace(0, 1, 128)
y = torch.linspace(0, 1, 128)
X_grid, Y_grid = torch.meshgrid(x, y, indexing='ij')
coords = torch.stack([X_grid.flatten(), Y_grid.flatten()], dim=1)
coords = 2.0 * coords - 1.0

In [ ]:
class DeepONetDataset(Dataset):
    def __init__(self, X_data, Y_data, coords, n_points=1000):
        self.X_data = torch.tensor(X_data, dtype=torch.float32)
        self.Y_data = torch.tensor(Y_data, dtype=torch.float32)
        self.coords = coords
        self.n_points = n_points

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):

        branch_input = self.X_data[idx]

        indices = torch.randint(0, 128*128, (self.n_points,))
        trunk_input = self.coords[indices]

        target_field = self.Y_data[idx].reshape(3, -1).permute(1,0)
        target = target_field[indices]

        return branch_input, trunk_input, target


In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, in_dim, mapping_size=64, scale=10.0):
        super().__init__()
        B = torch.randn(in_dim, mapping_size) * scale
        self.register_buffer("B", B)

    def forward(self, x):
        x_proj = 2 * torch.pi * x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

class BranchNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, latent_dim)

    def forward(self, u):
        features = self.encoder(u).squeeze(-1).squeeze(-1)
        return self.fc(features)


class TrunkNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fourier = FourierFeatures(2, mapping_size=64)

        self.net = nn.Sequential(
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, latent_dim * 3)
        )

        self.latent_dim = latent_dim

    def forward(self, x):
        x = self.fourier(x)
        out = self.net(x)
        return out.view(-1, 3, self.latent_dim)

class DeepONet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.branch = BranchNet(latent_dim)
        self.trunk = TrunkNet(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, u, x):
        B, n_pts, _ = x.shape

        branch_out = self.branch(u)
        trunk_out = self.trunk(x.view(-1, 2))
        trunk_out = trunk_out.view(B, n_pts, 3, self.latent_dim)

        branch_out = branch_out.unsqueeze(1).unsqueeze(2)

        output = torch.sum(branch_out * trunk_out, dim=-1)
        return output

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(target - pred, dim=(1,2))
    den = torch.norm(target, dim=(1,2))
    return (num / (den + 1e-8)).mean()

# Training

In [ ]:
import os

In [ ]:
data_path = "/content/drive/MyDrive/LDC_data"
geometries = ["harmonics", "nurbs", "skelneton"]

for geometry in geometries:

    wandb.init(
        project="DeepONet_LDC_uvp_ff",
        name=f"Improved_DeepONet_{geometry}",
        reinit=True,
        config={
            "epochs": 100,
            "batch_size": 8,
            "lr": 1e-3,
            "latent_dim": 256,
            "n_points": 1000
        }
    )


    X_data = np.load(os.path.join(data_path, f"{geometry}_lid_driven_cavity_X.npz"))['data']
    Y_data = np.load(os.path.join(data_path, f"{geometry}_lid_driven_cavity_Y.npz"))['data'][:,0:3]


    X_mean, X_std = X_data.mean(), X_data.std()
    Y_mean, Y_std = Y_data.mean(), Y_data.std()

    X_data = (X_data - X_mean) / (X_std + 1e-8)
    Y_data = (Y_data - Y_mean) / (Y_std + 1e-8)

    dataset = DeepONetDataset(X_data, Y_data, coords, wandb.config.n_points)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset,
                              batch_size=wandb.config.batch_size,
                              shuffle=True)

    val_loader = DataLoader(val_dataset,
                            batch_size=wandb.config.batch_size)

    model = DeepONet(wandb.config.latent_dim).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=wandb.config.lr)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=wandb.config.epochs
    )

    loss_fn = nn.MSELoss()


    for epoch in range(wandb.config.epochs):

        model.train()
        train_loss = 0

        for branch_input, trunk_input, target in train_loader:

            branch_input = branch_input.to(device)
            trunk_input = trunk_input.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            pred = model(branch_input, trunk_input)

            loss = loss_fn(pred, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        scheduler.step()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        val_l2 = 0

        with torch.no_grad():
            for branch_input, trunk_input, target in val_loader:

                branch_input = branch_input.to(device)
                trunk_input = trunk_input.to(device)
                target = target.to(device)

                pred = model(branch_input, trunk_input)

                val_loss += loss_fn(pred, target).item()
                val_l2 += relative_l2(pred, target).item()

        val_loss /= len(val_loader)
        val_l2 /= len(val_loader)

        wandb.log({
            "Epoch": epoch+1,
            "Train Loss": train_loss,
            "Val Loss": val_loss,
            "Val Relative L2": val_l2,
            "LR": scheduler.get_last_lr()[0]
        })

        print(f"Epoch {epoch+1} | Train {train_loss:.6f} | "
              f"Val {val_loss:.6f} | L2 {val_l2:.6f}")

    wandb.finish()

Epoch 1 | Train 0.998874 | Val 1.023460 | L2 0.958417
Epoch 2 | Train 0.992948 | Val 1.044063 | L2 0.951717
Epoch 3 | Train 0.987832 | Val 1.043429 | L2 0.955845
Epoch 4 | Train 0.985175 | Val 1.016504 | L2 0.962158
Epoch 5 | Train 0.995634 | Val 1.031570 | L2 0.946866
Epoch 6 | Train 0.990400 | Val 1.036456 | L2 0.940495
Epoch 7 | Train 0.983474 | Val 1.033681 | L2 0.935915
Epoch 8 | Train 0.984786 | Val 1.027999 | L2 0.930307
Epoch 9 | Train 0.984762 | Val 1.019058 | L2 0.948714
Epoch 10 | Train 0.985317 | Val 1.019535 | L2 0.942360
Epoch 11 | Train 0.984246 | Val 1.023467 | L2 0.929268
Epoch 12 | Train 0.983425 | Val 1.019287 | L2 0.928070
Epoch 13 | Train 0.983950 | Val 1.022271 | L2 0.926623
Epoch 14 | Train 0.980406 | Val 1.029354 | L2 0.928637
Epoch 15 | Train 0.981764 | Val 1.017172 | L2 0.938809
Epoch 16 | Train 0.983658 | Val 1.017398 | L2 0.942043
Epoch 17 | Train 0.983181 | Val 1.018790 | L2 0.923236
Epoch 18 | Train 0.980606 | Val 1.017849 | L2 0.930149
Epoch 19 | Train 0.

Epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███
LR,████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
Train Loss,█▆▅▇▅▃▄▄▄▄▄▃▅▄▆▃▄▃▂▃▃▃▃▃▃▂▂▃▃▂▁▃▁▂▂▂▂▁▂▂
Val Loss,▃█▅▆▄▂▃▂▃▅▂▁▃▅▂▄▂▃▂▁▃▂▁▃▃▃▂▂▃▁▂▂▂▂▃▂▃▂▃▂
Val Relative L2,█▇█▆▄▄▄▅▃▄▅▃▃▂▄▅▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,100
LR,0
Train Loss,0.97364
Val Loss,1.01711
Val Relative L2,0.90025


Epoch 1 | Train 1.009956 | Val 1.036875 | L2 1.031713
Epoch 2 | Train 1.001032 | Val 1.036974 | L2 1.053514
Epoch 3 | Train 0.998593 | Val 1.034062 | L2 1.023477
Epoch 4 | Train 1.000378 | Val 1.025942 | L2 1.005420
Epoch 5 | Train 0.998542 | Val 1.021419 | L2 1.048561
Epoch 6 | Train 0.994034 | Val 1.031624 | L2 1.025937
Epoch 7 | Train 0.994887 | Val 1.042490 | L2 1.111349
Epoch 8 | Train 0.995053 | Val 1.030989 | L2 1.016240
Epoch 9 | Train 0.996129 | Val 1.036060 | L2 1.077848
Epoch 10 | Train 0.991957 | Val 1.029864 | L2 1.007603
Epoch 11 | Train 0.990514 | Val 1.054722 | L2 1.238622
Epoch 12 | Train 0.993668 | Val 1.039737 | L2 1.089580
Epoch 13 | Train 0.992278 | Val 1.034029 | L2 1.032589
Epoch 14 | Train 0.990344 | Val 1.030005 | L2 1.013372
Epoch 15 | Train 0.985852 | Val 1.029029 | L2 1.013703
Epoch 16 | Train 0.992168 | Val 1.028924 | L2 0.997933
Epoch 17 | Train 0.992591 | Val 1.026792 | L2 1.014317
Epoch 18 | Train 0.992476 | Val 1.034584 | L2 1.023191
Epoch 19 | Train 0.

Epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
LR,██████████▇▇▇▇▇▇▆▆▆▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
Train Loss,▇█▄▃▁▄▃▅▄▄▄▃▅▃▅▄▃▂▄▃▄▄▁▃▄▃▄▄▂▄▃▃▂▂▁▃▂▄▂▂
Val Loss,▆▁▃█▃▇▄▃▂▂▅▃▃▃▂▁▁▂▂▃▂▄▂▃▁▂▃▂▁▂▃▃▂▂▃▃▃▂▂▂
Val Relative L2,█▇▄▃▄▇▁▂▂▃▁▂▂▁▄▃▂▄▅▅▄▆▅▃▅▅▅▇▅▄▄▄▅▅▅▅▅▅▅▆
Epoch,100
LR,0
Train Loss,0.98837
Val Loss,1.02918
Val Relative L2,1.02889


Epoch 1 | Train 0.942248 | Val 1.275474 | L2 1.078589
Epoch 2 | Train 0.939915 | Val 1.268726 | L2 1.561570
Epoch 3 | Train 0.946145 | Val 1.248759 | L2 1.613752
Epoch 4 | Train 0.951045 | Val 1.257680 | L2 2.276951
Epoch 5 | Train 0.933070 | Val 1.267232 | L2 1.760263
Epoch 6 | Train 0.944252 | Val 1.267179 | L2 1.551441
Epoch 7 | Train 0.939034 | Val 1.279075 | L2 1.129299
Epoch 8 | Train 0.935890 | Val 1.279278 | L2 2.562822
Epoch 9 | Train 0.940656 | Val 1.249260 | L2 1.521201
Epoch 10 | Train 0.945333 | Val 1.246335 | L2 1.197185
Epoch 11 | Train 0.932595 | Val 1.258234 | L2 1.341503
Epoch 12 | Train 0.930376 | Val 1.265760 | L2 1.125976
Epoch 13 | Train 0.928034 | Val 1.256920 | L2 1.130700
Epoch 14 | Train 0.944542 | Val 1.252845 | L2 1.074578
Epoch 15 | Train 0.930452 | Val 1.251371 | L2 1.040670
Epoch 16 | Train 0.934401 | Val 1.255947 | L2 1.175807
Epoch 17 | Train 0.941857 | Val 1.278638 | L2 1.342833
Epoch 18 | Train 0.939978 | Val 1.249770 | L2 1.092536
Epoch 19 | Train 0.

Epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
LR,██████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
Train Loss,▄▃▇▆█▅▄▄▅▅▄▆▄▃▅▅▅▄▄▄▆▃▃▄▄▆▄▁▃▅▃▂▂█▄▄▄▄▇▄
Val Loss,▆▃▄▇▃▄▃▄▃▂▅▄▆▅▅▄▅▆▇▅▁▃▆▅▃█▆▆▄▄▆▄▂▂▆▂▆▁▅▇
Val Relative L2,▁█▄▂▁▂▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▂▂▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂
Epoch,100
LR,0
Train Loss,0.93256
Val Loss,1.29179
Val Relative L2,1.24091


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
